# ResLib Tutorial: Supercharged LoRA with Reservoir Computing & MoE

This notebook demonstrates how to use **ResLib** for fast LLM adaptation using **Res-MoELoRA**. 
We will cover:
1. Installation
2. Supervised Fine-Tuning (SFT)
3. Group Relative Policy Optimization (GRPO)

## 1. Installation
First, we'll install the necessary dependencies and ResLib itself.

In [ ]:
!pip install torch transformers peft trl datasets accelerate bitsandbytes
!git clone https://github.com/MARVserver/ResLib.git # Replace with actual repo
%cd reslib
!pip install -e . --no-build-isolation

## 2. Supervised Fine-Tuning (SFT)
SFT is the process of fine-tuning a pre-trained model on a dataset of instruction-response pairs.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer
from reslib import ResMoELoRAConfig, inject_res_moelora

model_id = "facebook/opt-125m"
dataset_name = "timdettmers/openassistant-guanaco"

# Load Model
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Inject Res-MoELoRA
config = ResMoELoRAConfig(reservoir_size=128, num_experts=4, target_modules=["q_proj", "v_proj"])
model = inject_res_moelora(model, config)

# Dataset
dataset = load_dataset(dataset_name, split="train[:500]")

# Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=TrainingArguments(
        output_dir="./res_sft",
        per_device_train_batch_size=4,
        learning_rate=2e-4,
        num_train_epochs=1,
        fp16=True,
        report_to="none"
    ),
)
trainer.train()

## 3. Group Relative Policy Optimization (GRPO)
GRPO is a reinforcement learning algorithm used for aligning models with human preferences or specific rewards.

In [ ]:
from trl import GRPOTrainer, GRPOConfig

def length_reward_func(prompts, completions, **kwargs):
    return [float(len(c)) / 100.0 for c in completions]

dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train[:100]")
dataset = dataset.map(lambda x: {"prompt": x["prompt"]})

trainer = GRPOTrainer(
    model=model,
    reward_funcs=length_reward_func,
    args=GRPOConfig(
        output_dir="./res_grpo",
        per_device_train_batch_size=2,
        num_generations=4,
        max_prompt_length=128,
        max_completion_length=128,
        report_to="none"
    ),
    train_dataset=dataset,
)
trainer.train()